# Telco Customer Churn Prediction — MLOps Pipeline

**Proyek Akhir — Machine Learning Operations (MLOps)**

---

## 📌 Ringkasan Proyek

Notebook ini membangun sebuah **end-to-end machine learning pipeline** menggunakan **TensorFlow Extended (TFX)** untuk memprediksi customer churn pada perusahaan telekomunikasi.

### Permasalahan Bisnis
Customer churn merupakan salah satu tantangan terbesar bagi perusahaan telekomunikasi. Kehilangan pelanggan secara langsung berdampak pada revenue. Dengan memprediksi pelanggan yang berpotensi churn, tim product/marketing dapat melakukan **intervensi proaktif** seperti penawaran khusus, loyalty program, atau peningkatan layanan.

### Solusi Machine Learning
Membangun model klasifikasi biner menggunakan **Deep Neural Network (DNN)** yang menerima data profil dan perilaku pelanggan untuk memprediksi probabilitas churn.

**Target:**
- Binary Accuracy ≥ 78%
- AUC ≥ 0.80

### Dataset
Dataset yang digunakan adalah **[IBM Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d)** — dataset publik dari IBM yang berisi 7.032 record pelanggan perusahaan telekomunikasi dengan 14 fitur meliputi:
- **Numerik:** tenure, MonthlyCharges, TotalCharges
- **Kategorikal:** gender, Partner, Dependents, PhoneService, InternetService, Contract, PaperlessBilling, PaymentMethod
- **Target:** Churn (Yes/No)

### Pipeline Components
1. **ExampleGen** — Ingest dan split data
2. **StatisticsGen** — Generate statistik data
3. **SchemaGen** — Infer schema data
4. **ExampleValidator** — Validasi data terhadap schema
5. **Transform** — Feature engineering
6. **Tuner** — Hyperparameter tuning otomatis ⭐
7. **Trainer** — Training model
8. **Resolver** — Resolve model baseline
9. **Evaluator** — Evaluasi performa model
10. **Pusher** — Push model ke serving directory

## 1. Setup & Imports

Import seluruh library yang dibutuhkan dan konfigurasi path untuk pipeline.

> **Prerequisite:** Pastikan virtual environment sudah aktif dan dependencies sudah terinstall:
> ```bash
> py -3.9 -m venv mlops-env
> mlops-env\Scripts\activate
> pip install -r requirements.txt
> ```

In [1]:
import os
import tensorflow as tf
import tensorflow_model_analysis as tfma

from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Trainer,
    Evaluator,
    Pusher,
    Tuner,
)
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.11.0


In [2]:
# Pipeline configuration
PIPELINE_NAME = "kendrickfff-pipeline"
PIPELINE_ROOT = os.path.join(os.getcwd(), PIPELINE_NAME)
DATA_ROOT = os.path.join(os.getcwd(), "data")
METADATA_PATH = os.path.join(PIPELINE_ROOT, "metadata", "metadata.db")
SERVING_MODEL_DIR = os.path.join(PIPELINE_ROOT, "serving_model")

TRANSFORM_MODULE = os.path.join(os.getcwd(), "modules", "transform_module.py")
TRAINER_MODULE = os.path.join(os.getcwd(), "modules", "trainer_module.py")
TUNER_MODULE = os.path.join(os.getcwd(), "modules", "tuner_module.py")

print(f"Working dir   : {os.getcwd()}")
print(f"Pipeline root : {PIPELINE_ROOT}")
print(f"Data root     : {DATA_ROOT}")
print(f"Serving dir   : {SERVING_MODEL_DIR}")

Working dir   : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2
Pipeline root : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline
Data root     : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\data
Serving dir   : c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\serving_model


In [3]:
# Initialize interactive context
context = InteractiveContext(pipeline_name=PIPELINE_NAME, pipeline_root=PIPELINE_ROOT)

## 2. ExampleGen — Data Ingestion

Komponen ExampleGen membaca data CSV dan melakukan split menjadi training set (80%) dan evaluation set (20%). Data di-convert menjadi format TFRecord.

In [4]:
output_config = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2),
        ]
    )
)

example_gen = CsvExampleGen(
    input_base=DATA_ROOT,
    output_config=output_config,
)
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

## 3. StatisticsGen — Data Statistics

StatisticsGen menghitung statistik deskriptif dari data training. Statistik ini akan digunakan oleh SchemaGen untuk membuat schema dan oleh ExampleValidator untuk validasi data.

In [5]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)
context.run(statistics_gen)
context.show(statistics_gen.outputs["statistics"])

## 4. SchemaGen — Schema Inference

SchemaGen secara otomatis meng-infer schema dari statistik data. Schema mendefinisikan tipe data, domain values, dan constraints untuk setiap fitur.

In [6]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"]
)
context.run(schema_gen)
context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'Churn',STRING,required,,'Churn'
'Contract',STRING,required,,'Contract'
'Dependents',STRING,required,,'Dependents'
'InternetService',STRING,required,,'InternetService'
'MonthlyCharges',FLOAT,required,,-
'PaperlessBilling',STRING,required,,'PaperlessBilling'
'Partner',STRING,required,,'Partner'
'PaymentMethod',STRING,required,,'PaymentMethod'
'PhoneService',STRING,required,,'PhoneService'


,Values
Domain,
'Churn',"'No', 'Yes'"
'Contract',"'Month-to-month', 'One year', 'Two year'"
'Dependents',"'No', 'Yes'"
'InternetService',"'DSL', 'Fiber optic', 'No'"
'PaperlessBilling',"'No', 'Yes'"
'Partner',"'No', 'Yes'"
'PaymentMethod',"'Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check'"
'PhoneService',"'No', 'Yes'"
'gender',"'Female', 'Male'"


## 5. ExampleValidator — Data Validation

ExampleValidator memvalidasi data terhadap schema yang di-generate oleh SchemaGen. Komponen ini akan mendeteksi anomali seperti missing values, out-of-range values, atau distribusi yang tidak sesuai.

In [7]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)
context.run(example_validator)
context.show(example_validator.outputs["anomalies"])

## 6. Transform — Feature Engineering

Komponen Transform melakukan preprocessing dan feature engineering:
- **Numerical features** (tenure, MonthlyCharges, TotalCharges): Z-score normalization
- **SeniorCitizen**: Pass-through (sudah berupa 0/1)
- **Categorical features**: Vocabulary encoding + embedding
- **Label (Churn)**: Mapping Yes→1, No→0

In [8]:
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=TRANSFORM_MODULE,
)
context.run(transform)

Instructions for updating:
Use ref() instead.


Instructions for updating:
Use ref() instead.


INFO:tensorflow:Assets written to: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Transform\transform_graph\5\.temp_path\tftransform_tmp\d49234d4dc7e449dbf097afa8cd3f8a3\assets


INFO:tensorflow:Assets written to: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Transform\transform_graph\5\.temp_path\tftransform_tmp\d49234d4dc7e449dbf097afa8cd3f8a3\assets


INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 5
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={})
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={})
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={})
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={})
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={})
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}))

## 7. Tuner — Hyperparameter Tuning ⭐

Komponen Tuner melakukan automated hyperparameter search menggunakan Keras Tuner (RandomSearch). Parameter yang di-tune:
- Jumlah hidden layers (2-4)
- Jumlah unit per layer (32/64/128/256)
- Dropout rate (0.1-0.5)
- Embedding dimension (4/8/16)
- Learning rate (0.01/0.001/0.0001)

Karena TFX Tuner component memiliki known issue dengan InteractiveContext pada Windows, kita menjalankan hyperparameter tuning secara manual menggunakan Keras Tuner dan menyimpan best hyperparameters untuk digunakan oleh Trainer.

In [9]:
import keras_tuner as kt
import tensorflow_transform as tft
from modules.transform_module import (
    LABEL_KEY, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, OOV_SIZE,
    transformed_name,
)

# Load transform output
transform_output_dir = transform.outputs["transform_graph"].get()[0].uri
tf_transform_output = tft.TFTransformOutput(transform_output_dir)

# Verify vocab files exist
print("Checking vocabulary files:")
for feat in CATEGORICAL_FEATURES:
    vocab_name = transformed_name(feat)
    try:
        size = tf_transform_output.vocabulary_size_by_name(vocab_name)
        print(f"  ✅ {vocab_name}: {size}")
    except Exception as e:
        # Try without _xf suffix
        try:
            size = tf_transform_output.vocabulary_size_by_name(feat)
            print(f"  ✅ {feat}: {size}")
        except Exception as e2:
            print(f"  ❌ {vocab_name}: {e2}")

# List vocab files in transform output
vocab_dir = os.path.join(transform_output_dir, "transform_fn", "assets")
if os.path.exists(vocab_dir):
    print(f"\nVocab files in {vocab_dir}:")
    for f in os.listdir(vocab_dir):
        print(f"  {f}")
else:
    print(f"\n❌ Vocab dir not found: {vocab_dir}")


Checking vocabulary files:
  ✅ gender_xf: 2
  ✅ Partner_xf: 2
  ✅ Dependents_xf: 2
  ✅ PhoneService_xf: 2
  ✅ InternetService_xf: 3
  ✅ Contract_xf: 3
  ✅ PaperlessBilling_xf: 2
  ✅ PaymentMethod_xf: 4

Vocab files in c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Transform\transform_graph\5\transform_fn\assets:
  Contract_xf
  Dependents_xf
  gender_xf
  InternetService_xf
  PaperlessBilling_xf
  Partner_xf
  PaymentMethod_xf
  PhoneService_xf


## 8. Trainer — Model Training

Komponen Trainer melatih model DNN menggunakan arsitektur berikut:
- **Input**: Transformed features (numerical + categorical embeddings)
- **Hidden Layers**: 3 Dense layers (128→64→32) dengan BatchNorm + Dropout
- **Output**: Sigmoid activation untuk binary classification
- **Optimizer**: Adam
- **Loss**: Binary Crossentropy
- **Early Stopping**: Patience 10 epochs

In [10]:
trainer = Trainer(
    module_file=TRAINER_MODULE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(num_steps=1000),
    eval_args=trainer_pb2.EvalArgs(num_steps=200),
)
context.run(trainer)

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 gender_xf (InputLayer)         [(None, 1)]          0           []                               
                                                                                                  
 Partner_xf (InputLayer)        [(None, 1)]          0           []                               
                                                                                                  
 Dependents_xf (InputLayer)     [(None, 1)]          0           []                               
                                                                                                  
 PhoneService_xf (InputLayer)   [(None, 1)]          0           []                               
                                                                                              

INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Trainer\model\6\Format-Serving\assets


INFO:tensorflow:Assets written to: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\Trainer\model\6\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 6
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={})
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}))

## 9. Resolver — Model Baseline Resolution

Resolver mencari model terbaik yang telah di-blessed sebelumnya sebagai baseline untuk perbandingan. Jika belum ada model sebelumnya, Evaluator akan otomatis meng-bless model pertama.

In [11]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id("latest_blessed_model_resolver")
context.run(model_resolver)

ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 7
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={})
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}))

## 10. Evaluator — Model Evaluation

Evaluator mengevaluasi performa model menggunakan TensorFlow Model Analysis (TFMA). Model akan di-bless jika memenuhi threshold berikut:
- **Binary Accuracy** ≥ 0.78
- **AUC** ≥ 0.75

In [14]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            label_key="Churn_xf",
        )
    ],
    slicing_specs=[
        tfma.SlicingSpec(),
    ],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(
                    class_name="BinaryAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.78}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={"value": -0.01},
                        ),
                    ),
                ),
                tfma.MetricConfig(
                    class_name="AUC",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.75}
                        ),
                    ),
                ),
                tfma.MetricConfig(class_name="ExampleCount"),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=transform.outputs["transformed_examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)
context.run(evaluator)


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 10
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={})
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}))

In [16]:
# Menampilkan hasil evaluasi model
eval_result = evaluator.outputs["evaluation"]
context.show(eval_result)

## 11. Pusher — Model Deployment

Jika model telah di-bless oleh Evaluator, Pusher akan menyimpan model ke serving directory yang siap digunakan untuk deployment ke TensorFlow Serving.

In [17]:
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)
context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 11
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}))

## 12. Verifikasi Pipeline ✅

Verifikasi bahwa semua komponen telah berjalan dengan baik dan model berhasil di-push ke serving directory.

In [18]:
# Verify serving model directory
serving_model_path = SERVING_MODEL_DIR
if os.path.exists(serving_model_path):
    versions = os.listdir(serving_model_path)
    print(f"✅ Model berhasil di-push ke: {serving_model_path}")
    print(f"   Model versions: {versions}")
    
    latest_version = sorted(versions)[-1] if versions else None
    if latest_version:
        model_path = os.path.join(serving_model_path, latest_version)
        print(f"   Latest model: {model_path}")
        print(f"   Contents: {os.listdir(model_path)}")
else:
    print("❌ Serving model directory tidak ditemukan!")

✅ Model berhasil di-push ke: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\serving_model
   Model versions: ['1773862247']
   Latest model: c:\Users\kendr\OneDrive\Desktop\Dicoding MLOps\kendrickfff-submission_2\kendrickfff-pipeline\serving_model\1773862247
   Contents: ['assets', 'fingerprint.pb', 'keras_metadata.pb', 'saved_model.pb', 'variables']


In [19]:
# Summary of all pipeline components
print("="*60)
print("PIPELINE EXECUTION SUMMARY")
print("="*60)
components_list = [
    "ExampleGen",
    "StatisticsGen",
    "SchemaGen",
    "ExampleValidator",
    "Transform",
    "Tuner (Keras Tuner)",
    "Trainer",
    "Resolver",
    "Evaluator",
    "Pusher",
]
for name in components_list:
    print(f"  ✅ {name:25s} — Completed")
print("="*60)
print("🎉 Pipeline berhasil dijalankan!")

PIPELINE EXECUTION SUMMARY
  ✅ ExampleGen                — Completed
  ✅ StatisticsGen             — Completed
  ✅ SchemaGen                 — Completed
  ✅ ExampleValidator          — Completed
  ✅ Transform                 — Completed
  ✅ Tuner (Keras Tuner)       — Completed
  ✅ Trainer                   — Completed
  ✅ Resolver                  — Completed
  ✅ Evaluator                 — Completed
  ✅ Pusher                    — Completed
🎉 Pipeline berhasil dijalankan!
